In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from src.utils import compute_individual_expectation_values, build_hva_layers, build_sparse_hamiltonian,find_best_initial_params,test_energy_consistency,optimize_energy, compute_error

from qiskit.providers.fake_provider import GenericBackendV2

from matplotlib import pyplot as plt
import sys
from qiskit.transpiler import CouplingMap


sys.path.append("../../pauli_lindblad_per/")
from tomography.experiment import SparsePauliTomographyExperiment as tomography

plt.style.use("ggplot")

In [2]:
# Lineare Kopplung: 0–1–2–3
coupling = CouplingMap(couplinglist=[(0, 1), (1, 2), (2, 3)])
backend = GenericBackendV2(num_qubits=4, coupling_map=coupling)

hx = 1
hz = 1
J=-1
H_sparse = build_sparse_hamiltonian(hx, hz, backend, J=J)

best_guess, energy_exact = find_best_initial_params(H_sparse, backend)

opt_params, min_error = optimize_energy(best_guess, H_sparse, backend, energy_exact)

print("Optimierte Parameter :", opt_params)
print("Minimaler Fehler     :", min_error)

expectation_values, paulis = compute_individual_expectation_values(opt_params, backend, hx, hz,J=J, num_layers=1)
pauli_list = paulis
print(pauli_list)

Optimierte Parameter : [-3.14159265 -1.57071519  1.57085478]
Minimaler Fehler     : 0.1048468493862667
['IIZZ', 'IZZI', 'ZZII', 'IIIX', 'IIIZ', 'IIXI', 'IIZI', 'IXII', 'IZII', 'XIII', 'ZIII']


In [3]:
inst_map = [0,1,2,3]
qc = build_hva_layers(opt_params, backend, num_layers=1)

qc.draw(fold=-1)

┌───┐┌─────────────┐┌────────────┐                                                            
q_0: ┤ H ├┤ Rz(-1.5707) ├┤ Rx(1.5709) ├──■──────────────■──────────────────────────────────────────
     ├───┤├─────────────┤├────────────┤┌─┴─┐┌────────┐┌─┴─┐                                        
q_1: ┤ H ├┤ Rz(-1.5707) ├┤ Rx(1.5709) ├┤ X ├┤ Rz(-π) ├┤ X ├──■──────────────■──────────────────────
     ├───┤├─────────────┤├────────────┤└───┘└────────┘└───┘┌─┴─┐┌────────┐┌─┴─┐                    
q_2: ┤ H ├┤ Rz(-1.5707) ├┤ Rx(1.5709) ├────────────────────┤ X ├┤ Rz(-π) ├┤ X ├──■──────────────■──
     ├───┤├─────────────┤├────────────┤                    └───┘└────────┘└───┘┌─┴─┐┌────────┐┌─┴─┐
q_3: ┤ H ├┤ Rz(-1.5707) ├┤ Rx(1.5709) ├────────────────────────────────────────┤ X ├┤ Rz(-π) ├┤ X ├
     └───┘└─────────────┘└────────────┘                                        └───┘└────────┘└───┘

In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

service = QiskitRuntimeService(channel="ibm_quantum", token="YOUR_TOKEN")

backend = service.get_backend("ibm_aachen")
sampler = SamplerV2(backend=backend)

RequestsApiError: 'HTTPSConnectionPool(host=\'auth.quantum.ibm.com\', port=443): Max retries exceeded with url: /api/version (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000020903FAA6E0>: Failed to resolve \'auth.quantum.ibm.com\' ([Errno 11001] getaddrinfo failed)"))'

In [5]:
from src.calculate_runs import count_tomography_runs_from_repo, count_per_runs

# TOMOGRAPHY
res_tomo = count_tomography_runs_from_repo(
    qc=qc,
    inst_map=inst_map,
    backend=backend,
    used_qubits=inst_map,
    depths=[2,4,16,32,64],
    samples=100,
    single_samples=250,
    shots=1024   # optional
)
print("Tomography: ",res_tomo['backend_runs'])

# PER
res_per = count_per_runs(
    qc=qc,
    pauli_list=pauli_list,
    noise_strengths=[0,1,2],
    samples=1000,
    shots=1024   # optional
)
print("PER: ",res_per['backend_runs'])
print("Total: ",res_per['backend_runs'] + res_tomo['backend_runs'])

Tomography:  18432000
PER:  15360000
Total:  33792000


In [ ]:
experiment = tomography(
    circuits = [qc], 
    inst_map = inst_map,
    backend = backend, 
    used_qubits= inst_map
    )

experiment.generate(
    samples = 100, 
    single_samples = 250, 
    depths = [2,4,16,32,64] 
    )

shots = 1024
def executor(circuits):
    job = sampler.run(circuits, shots=shots)
    result = job.result()
    counts_list = []
    for pub_result in result.pubs:
        counts = pub_result.data.c.get_counts()
        counts_list.append(counts)
        
    return counts_list

#run the experiment
experiment.run(executor)
noisedataframe = experiment.analyze()

In [ ]:
# 2. PER-Experiment Setup
perexp = experiment.create_per_experiment([qc])

perexp.generate(
    expectations=pauli_list,
    samples=1000,
    noise_strengths=[0, 1, 2]
)

# 3. Ausführen
perexp.run(executor)

# 4. Analysieren
per_results = perexp.analyze()

In [ ]:
# Compare absolute vs relative errors
for k in range(len(pauli_list)):
    exact = expectation_values[k]
    calc = per_results[0].get_result(pauli_list[k]).get_expectations()[0]
    abs_error = abs(exact - calc)
    rel_error = abs_error / abs(exact) if exact != 0 else float('inf')
    
    print(f"{pauli_list[k]}: Exact={exact:.4f}, Calc={calc:.4f}")
    print(f"  Absolute error: {abs_error:.6f}")
    print(f"  Relative error: {rel_error*100:.2f}%")